In [3]:
import numpy as np
import time
from collections import defaultdict

t0 = time.time()

# ── 1. Parse header ──────────────────────────────────────────────────────────
info = []
with open("LastConfig2.bfm", "r") as file:
    data = file.readlines()

for i, line in enumerate(data):
    if (line.startswith("!") or line.startswith("#!")) and "=" in line:
        textstring = line.strip()[1:]
        name  = textstring[:textstring.find("=")]
        value = float(textstring[textstring.find("=") + 1:])
        info.append([name, value, i])

# ── 2. Build named lookup (fixes Bug 3) ─────────────────────────────────────
info_dict  = {entry[0]: entry[1] for entry in info}
info_index = {entry[0]: entry[2] for entry in info}  # line index in file

number_of_monomers      = info_dict["number_of_monomers"]
box_x                   = info_dict["box_x"]
box_y                   = info_dict["box_y"]
box_z                   = info_dict["box_z"]
try:
    number_of_linear_chains = info_dict["number_of_linear_chains"]
    number_of_crosslinkers  = info_dict["number_of_crosslinkers"]
    chainLength             = info_dict["chainLength"]
    #mcs                     = info_dict[-1][1]
    #print(mcs)  # last entry
except KeyError as e:
    number_of_linear_chains = info_dict["!number_of_linear_chains"]
    number_of_crosslinkers  = info_dict["!number_of_crosslinkers"]
    chainLength             = info_dict["!chainLength"]

params = {
    "number_of_monomers":      number_of_monomers,
    "box_x":                   box_x,
    "box_y":                   box_y,
    "box_z":                   box_z,
    "number_of_linear_chains": number_of_linear_chains,
    "number_of_crosslinkers":  number_of_crosslinkers,
    "chainLength":             chainLength,
    #"mcs":                     mcs,
}

with open("system.txt", "w") as f:
    for key, value in params.items():
        f.write(f"{key} = {value}\n")

# ── 3. Parse bonds ───────────────────────────────────────────────────────────
bonds_raw = "".join(data[data.index("!bonds\n") + 1 : info_index["box_x"]])
bonds = np.fromstring(bonds_raw, sep=" ", dtype=float).reshape(-1, 2)

# Fix Bug 1: column order not guaranteed — sort each row so chain monomer
# (smaller, 1-based global index ≤ chainLength*N_chains) is always column 0
bonds = np.sort(bonds, axis=1)

In [4]:
chain_monomers = bonds[:, 0]   # global index of the chain endpoint monomer
xlink_globals  = bonds[:, 1]   # global index of the crosslinker monomer

chainsID = (chain_monomers - 1) // chainLength + 1          # 1-based chain ID
xlink    = xlink_globals - chainLength * number_of_linear_chains  # 1-based local xlink ID

bondss = np.column_stack([chainsID, xlink])
sorted_bonds = bondss[bondss[:, 0].argsort()]

# ── 5. Build chains → crosslinkers map ──────────────────────────────────────
chains = sorted_bonds[:, 0]
unique_chains, inverse = np.unique(chains, return_inverse=True)

chains_xlinks = []
for i, chain in enumerate(unique_chains):
    mask = inverse == i
    associated = sorted_bonds[mask, 1]

    if associated.size < 2:       # skip chains bonded to only 1 crosslinker
        print(f"Chain {chain} (ID: {i}) is bonded to only 1 crosslinker, skipping...")
        continue

    chains_xlinks.append([chain] + associated.tolist())

Chain 11.0 (ID: 10) is bonded to only 1 crosslinker, skipping...
Chain 119.0 (ID: 118) is bonded to only 1 crosslinker, skipping...
Chain 216.0 (ID: 215) is bonded to only 1 crosslinker, skipping...
Chain 239.0 (ID: 238) is bonded to only 1 crosslinker, skipping...
Chain 493.0 (ID: 492) is bonded to only 1 crosslinker, skipping...
Chain 506.0 (ID: 505) is bonded to only 1 crosslinker, skipping...
Chain 566.0 (ID: 565) is bonded to only 1 crosslinker, skipping...
Chain 569.0 (ID: 568) is bonded to only 1 crosslinker, skipping...
Chain 574.0 (ID: 573) is bonded to only 1 crosslinker, skipping...
Chain 643.0 (ID: 642) is bonded to only 1 crosslinker, skipping...
Chain 681.0 (ID: 680) is bonded to only 1 crosslinker, skipping...
Chain 842.0 (ID: 841) is bonded to only 1 crosslinker, skipping...
Chain 915.0 (ID: 914) is bonded to only 1 crosslinker, skipping...
Chain 973.0 (ID: 972) is bonded to only 1 crosslinker, skipping...
Chain 1196.0 (ID: 1195) is bonded to only 1 crosslinker, skippin

In [7]:
len(chains_xlinks)

119384